# Le fil rouge 2020-2025 : l'inflation s'envole, la Fed la poursuit · *The running thread 2020-2025: inflation soars, the Fed gives chase*

Notebook compagnon du chapitre **4. Les grandes variables macro qui font bouger vos placements : un panorama** — [lire l'article](https://nmlab.io/ressources/les-grandes-variables-macro-qui-font-bouger-vos-placements).
Companion notebook to chapter **4. The Big Macro Variables That Move Your Investments: A Panorama** — [read the article](https://nmlab.io/en/ressources/the-big-macro-variables-that-move-your-investments).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_data() -> tuple[Series, Series]:
    """Inflation américaine sur un an (IPC, CPIAUCSL) et borne haute du taux directeur
    de la Fed (DFEDTARU), en direct de FRED, depuis 2019.
    U.S. year-on-year inflation (CPI) and the Fed policy-rate upper bound, live from FRED."""
    cpi = nm.load_fred("CPIAUCSL", start="2018")
    inflation = (cpi.pct_change(12) * 100).loc["2019":]
    fedrate = nm.load_fred("DFEDTARU", start="2019")
    return inflation, fedrate

inflation, fedrate = load_data()


import matplotlib.dates as mdates
import pandas as pd
from pandas import Series
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Le fil rouge 2020-2025 : l'inflation s'envole, la Fed la poursuit",
        sub="États-Unis, 2019-2025 \u2014 inflation sur un an et borne haute du taux directeur",
        infl="inflation américaine\n(prix à la consommation,\nsur un an)",
        fed="taux directeur\nde la Fed",
        peak="pic : \u2248 9 %\njuin 2022", covid="COVID : taux ramenés\nprès de 0 %",
        hike="premier relèvement\nmars 2022", plateau="sommet : 5,25-5,50 %\njuillet 2023",
        cut="première baisse\nsept. 2024", target="cible d'inflation : 2 %",
        note="Source : Bureau of Labor Statistics et Réserve fédérale, via FRED (séries CPIAUCSL et DFEDTARU)."),
    "en": dict(
        title="The running thread 2020-2025: inflation soars, the Fed gives chase",
        sub="United States, 2019-2025 \u2014 year-on-year inflation and the policy-rate upper bound",
        infl="U.S. inflation\n(consumer prices,\nyear on year)",
        fed="Fed policy\nrate",
        peak="peak: \u2248 9%\nJune 2022", covid="COVID: rates cut\nto near 0%",
        hike="first hike\nMarch 2022", plateau="peak: 5.25-5.50%\nJuly 2023",
        cut="first cut\nSept. 2024", target="inflation target: 2%",
        note="Source: Bureau of Labor Statistics and Federal Reserve, via FRED (series CPIAUCSL and DFEDTARU)."),
}

def build_figure(inflation: Series, fedrate: Series, lang: str) -> Figure:
    """Inflation (rose) et taux directeur en escalier (bleu), avec les jalons du cycle 2020-2025."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1225)
    ax = nm.axes(fig)
    ax.axhline(2, color=nm.COLORS["amber"], linestyle=(0, (6, 4)), linewidth=2.2)
    ax.plot(fedrate.index, fedrate, color=nm.COLORS["blue"], linewidth=3.0, drawstyle="steps-post")
    ax.plot(inflation.index, inflation, color=nm.COLORS["rose"], linewidth=3.0)
    ax.set_ylim(-0.4, 10.8)
    ax.set_yticks(range(0, 11, 2))
    fmt = " %" if lang == "fr" else "%"
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}{fmt}"))
    ax.set_xlim(pd.Timestamp("2019-01-01"), pd.Timestamp("2026-06-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.text(pd.Timestamp("2021-01-15"), 8.1, text["infl"], fontsize=21.5, fontweight="bold",
            color=nm.COLORS["rose"], va="center", linespacing=1.4)
    ax.text(pd.Timestamp("2019-02-01"), 3.15, text["fed"], fontsize=21.5, fontweight="bold",
            color=nm.COLORS["blue"], va="center", linespacing=1.4)
    ax.text(ax.get_xlim()[1], 2.35, text["target"], fontsize=20, color=nm.COLORS["muted"],
            ha="right", va="bottom")

    peak_date = inflation.loc["2022"].idxmax()
    peak_val = float(inflation.loc[peak_date])
    ax.scatter([peak_date], [peak_val], s=95, color=nm.COLORS["rose"], zorder=5)
    ax.annotate(text["peak"], xy=(peak_date, peak_val), xytext=(peak_date, 10.4),
                ha="center", va="top", fontsize=20, color=nm.COLORS["text"], linespacing=1.4)

    covid_date = fedrate.loc["2020"].idxmin()
    ax.scatter([covid_date], [float(fedrate.loc[covid_date])], s=85, color=nm.COLORS["blue"], zorder=5)
    ax.annotate(text["covid"], xy=(covid_date, float(fedrate.loc[covid_date])),
                xytext=(covid_date, -0.15), ha="center", va="top",
                fontsize=20, color=nm.COLORS["text"], linespacing=1.4)

    hike_date = pd.Timestamp("2022-03-17")
    ax.scatter([hike_date], [float(fedrate.asof(hike_date))], s=85, color=nm.COLORS["blue"], zorder=5)
    ax.annotate(text["hike"], xy=(hike_date, float(fedrate.asof(hike_date))),
                xytext=(pd.Timestamp("2021-10-01"), 1.35), ha="center", va="center",
                fontsize=20, color=nm.COLORS["text"], linespacing=1.4)

    ax.text(pd.Timestamp("2023-08-01"), 6.15, text["plateau"], fontsize=20,
            color=nm.COLORS["text"], ha="center", va="center", linespacing=1.4)

    cut_date = pd.Timestamp("2024-09-18")
    ax.scatter([cut_date], [float(fedrate.asof(cut_date))], s=85, color=nm.COLORS["blue"], zorder=5)
    ax.annotate(text["cut"], xy=(cut_date, float(fedrate.asof(cut_date))),
                xytext=(pd.Timestamp("2025-02-01"), 6.4), ha="center", va="center",
                fontsize=20, color=nm.COLORS["text"], linespacing=1.4)

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(inflation, fedrate, LANG)